# 🕵️ Mission 01 — The Codebreaker's Computer

**Operation: Signals & Secrets · Companion to the Codebreaker's Kit**

Agent, you already cracked a code *by hand* using your cipher wheel and the frequency trick. That was the hard way — the way the codebreakers at Bletchley Park did it, with pencil and paper.

Now you'll teach a **computer** to do it in a blink.

By the end of this notebook you will:
1. Make the computer **encode** a secret message,
2. **Decode** one when you know the shift,
3. **Break** one when you *don't* — using the same frequency trick you used by hand,
4. Crack the **real intercept** from your Kit, automatically.

> 💡 **How to run a cell:** click it, then press **Shift + Enter**. Run them in order, top to bottom.


## Step 1 — Shift a single letter

Every Caesar cipher is built on one tiny move: take a letter and slide it forward in the alphabet by some number — the **shift**.

`A` shifted by 3 becomes `D`. `B` becomes `E`. And so on.

The tricky part is the end of the alphabet: `Z` shifted by 1 has to **wrap around** back to `A`. Computers do wrap-around with the `%` (modulo) operator — it's just "the remainder after dividing." `26 % 26` is `0`, which lands us right back at `A`.

Run this to see one letter move:

In [ ]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def shift_letter(letter, shift):
    # Where is this letter in the alphabet? (A=0, B=1, ... Z=25)
    position = ALPHABET.index(letter)
    # Slide it over, and wrap around using % 26
    new_position = (position + shift) % 26
    return ALPHABET[new_position]

# Try it! Change these and re-run.
print("A shifted by 3 :", shift_letter("A", 3))
print("Z shifted by 1 :", shift_letter("Z", 1))   # watch it wrap to A
print("E shifted by 7 :", shift_letter("E", 7))

## Step 2 — Encode a whole message

To scramble a whole sentence, we just shift **every letter** and leave spaces alone. The loop below walks through the message one character at a time.

This is exactly what your cipher wheel does — turn it once and read off every letter.

In [ ]:
def encode(message, shift):
    result = ""
    for char in message.upper():
        if char in ALPHABET:
            result += shift_letter(char, shift)
        else:
            result += char          # keep spaces and punctuation as-is
    return result

secret = encode("MEET AT THE OAK TREE", 5)
print(secret)

## Step 3 — Decode (when you know the shift)

If your partner tells you the secret shift, decoding is easy: shift **backwards** by the same amount. And here's a neat trick — shifting back by 5 is the same as shifting *forward* by `26 - 5 = 21`. So we can reuse the encode machine we already built!

In [ ]:
def decode(secret_message, shift):
    return encode(secret_message, 26 - shift)   # shift the other way around

print(decode(secret, 5))   # should read your original message again

## Step 4 — Break it the slow way: try every shift

But what if an enemy message arrives and you **don't** know the shift? 🤔

There are only 26 possible shifts. A computer can just try them **all** in a fraction of a second and let you eyeball which one turns into real English. This is called a **brute-force attack**.

In [ ]:
def brute_force(secret_message):
    for shift in range(26):
        print(f"shift {shift:2d} : {decode(secret_message, shift)}")

# An intercepted message — we have no idea what shift was used:
mystery = "AEMX MR XLI XVIIW"
brute_force(mystery)

👀 Scroll the results above and find the line that reads as real English. Which shift unlocked it?

Brute force works here because there are only 26 keys. But imagine a code with *trillions* of keys — like Enigma. Trying them all by hand was impossible, which is exactly why Bletchley Park had to **build a machine** (the Bombe). You're feeling the same problem they felt.

## Step 5 — Break it the smart way: frequency analysis

Now the real magic — let the computer find the shift **on its own**, the way you did with tally marks.

The plan is exactly your by-hand method:
1. Count how often each letter appears in the secret message.
2. The most common letter is *probably* a disguised **E** (the most common letter in English).
3. Work out the shift that turns **E** into that letter — that's the key.

In [ ]:
def crack(secret_message):
    # 1. Count each letter
    counts = {}
    for char in secret_message:
        if char in ALPHABET:
            counts[char] = counts.get(char, 0) + 1

    # 2. Find the most common letter
    most_common = max(counts, key=counts.get)

    # 3. If that letter is really 'E', what shift was used?
    #    shift = (where E went) - (where E started)
    guessed_shift = (ALPHABET.index(most_common) - ALPHABET.index("E")) % 26

    print(f"Most common letter : {most_common}")
    print(f"Best guess shift   : {guessed_shift}")
    print(f"Decoded message    : {decode(secret_message, guessed_shift)}")

# Test it on a message — the computer is never told the shift:
crack(encode("THE CODEBREAKER NEVER GUESSES THE KEY THE LETTERS GIVE IT AWAY", 11))

## Step 6 — 🎯 Crack the real intercept

This is the message from your Codebreaker's Kit — the one you (hopefully!) cracked by hand. Let's see the computer do it with no hints at all.

Run it and watch your secret summer orders appear.

In [ ]:
intercept = (
    "DLSJVTL HNLUA HBNNPL FVBY TPZZPVU "
    "AOPZ ZBTTLY PZ AV THZALY AOL "
    "ZLJYLA SPML VM TLZZHNLZ"
)

crack(intercept)

## 🧠 Shannon's twist — why this code can't survive

You just proved something important: a Caesar cipher is **weak**, because the same letter always hides the same way. The letter frequencies leak straight through the disguise, and a computer reads them instantly.

**Claude Shannon** — the patron of your whole summer — proved there's only one cipher that *never* leaks: the **one-time pad**, where the shift changes for every single letter using a random key as long as the message. With a true one-time pad, the counts come out flat, every letter equally likely. Run `crack()` on *that* and it finds nothing — there's no most-common letter to grab onto.

The catch (and Shannon proved this too): you and your partner both need the exact same random key, kept perfectly secret. Sharing keys safely is the hardest problem in all of cryptography — and it's the reason codes kept evolving all the way to the AES encryption inside the LoRa radios you'll build in Mission 04.

## 🏅 Agent challenges

Try these to earn your stripes:

1. **Send-back.** Encode a reply to your Handler with a shift of your choice and have them `crack()` it.
2. **Smarter cracker.** Right now we assume the top letter is `E`. Sometimes it's `T` or `A`. Can you make `crack()` print the **top 3** guesses instead of just one?
3. **Build a one-time pad.** Write a function that uses a *list* of shifts (one per letter) instead of a single number. Then try to `crack()` it — and watch the frequency trick fail. That failure *is* Shannon's perfect secrecy.

> When you're done here, report to your Handler. Mission 02 is waiting: making a signal you can *see*.
